A random field gives every position in space a random value, with values at nearby positions alike. Random fields describe shadow fading in radio channels, the hydraulic conductivity of aquifers, the height of an ocean surface and many other quantities that vary in space without a fixed pattern. This tutorial explains what they are, how they are described by their autocorrelation function, and how the sum-of-sinusoids method of Quadriga-Lib generates them at any position, in one, two or three dimensions.

The notebook needs Quadriga-Lib 0.13 or newer, numpy and matplotlib; see the setup on the [tutorial overview](tutorials.html#setup).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import quadriga_lib as qd

print(qd.version())

If the import fails or the version is older than 0.13, the notebook most likely runs in another Python environment than the one Quadriga-Lib is installed in: select that environment as the kernel, or update the package as described in the setup. A suffix `_AVX2` means that the fast vector code of the library is active.

## A first random field

Four lines give a map of 200 m × 200 m. The values change smoothly: values 10 m apart have a correlation of only 0.37, which makes 10 m the *decorrelation distance* of the generator; patches of high or low values are about twice that size. The terms used here, a *preset* and a *seed*, are explained further down.

In [ ]:
x = np.arange(0.0, 200.0, 0.5)                       # grid points in m, x and y
freq, acf, dist = qd.sos.preset("Comb256", 10.0)     # frequencies [256, 3] in 1/m for a decorrelation distance of 10 m
phase = qd.sos.init_phases(freq.shape[0], 1, 42)     # phases [256, 1] in rad: one generator, seed 42
field = qd.sos.map(freq, phase, x, x)[:, :, 0, 0]    # values on the grid, rows are y, columns are x

plt.imshow(field, origin="lower", extent=[0, 200, 0, 200], cmap=plt.get_cmap("viridis", 32))
plt.colorbar(label="value")
plt.xlabel("x in m"); plt.ylabel("y in m")
plt.show()

## What a random field is

A random field is a random function of position [9, Sec. 1]: at every position $\mathbf{x}$ it has a random value $s(\mathbf{x})$, and the values at different positions are related. A random process is the special case of a one-dimensional index, usually time. The map above is one *realization* of the field; another seed gives another map with the same statistics. Statements such as "the values are Normal distributed with zero mean and unit variance" are about the *ensemble*, the set of all realizations, not about one map.

Two simplifications are common and used here. A field is *stationary* (homogeneous) if its statistics are the same everywhere, and *isotropic* if the relation between two positions depends only on their distance, not on the direction [1, Sec. 4.1].

Random fields appear wherever a quantity varies in space without a fixed pattern:

- Radio channels: shadow fading, the slow change of the received power behind buildings and trees, and the other large-scale parameters of channel models such as the 3GPP model [2], [3].
- Geostatistics and hydrology: the hydraulic conductivity of aquifers [4].
- Ocean waves: the height of the sea surface, built from many waves [5].
- Optics: the phase disturbance of light passing through turbulent air [6].
- Materials: two-phase composite materials such as alloys and microemulsions, modeled as level cuts of a Gaussian random field [7].

## The autocorrelation function

How quickly the values change with the position is described by the autocorrelation function (ACF). For a stationary, isotropic field with zero mean and unit variance it is the expected product of the values at two positions a distance $d$ apart,

$$\rho(d) = \mathrm{E}\{\, s(\mathbf{x})\, s(\mathbf{x} + \mathbf{d}) \,\}, \qquad |\mathbf{d}| = d .$$

It starts at $\rho(0) = 1$ and falls towards 0 for positions far apart. The *decorrelation distance* $d_0$ is the distance at which it has fallen to $1/e \approx 0.37$ [2]; this is the correlation distance of the exponential ACF of 3GPP [3, clause 7.4.4]. The exponential decay of the shadow-fading correlation with distance was found in measurements by Gudmundson [8], who described it by the correlation at a reference distance instead, e.g. 0.82 at 100 m. Other sources use other lengths for the same idea, such as the distance at which the correlation falls to 1/2 [14], the length parameter of a kernel [1, Sec. 4.2], or the integral scale in geostatistics [13]; when comparing numbers from other sources, check which one is meant.

Quadriga-Lib has three ACF shapes. `Exp` is the exponential $\rho(d) = e^{-d/d_0}$ of 3GPP [3, clause 7.4.4]. `Gauss` is $\rho(d) = e^{-(d/d_0)^2}$. `Comb` follows `Gauss` up to $d_0$ and `Exp` beyond.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))
for name, style in (("Exp256", "-"), ("Gauss256", "-"), ("Comb256", "--")):
    f, a, d = qd.sos.preset(name, 10.0)
    ax.plot(d, a, style, lw=2 if style == "--" else 1.5, label=name[:-3])
ax.axhline(np.exp(-1), color="gray", lw=0.8, ls="--")
ax.axvline(10.0, color="gray", lw=0.8, ls="--")
ax.set_xlim(0, 40); ax.set_xlabel("distance d in m"); ax.set_ylabel("ACF"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

The shape near $d = 0$ decides how smooth the field is. The exponential ACF has a kink at the origin: the field is continuous but rough, it changes even over very short distances. The Gaussian ACF is flat at the origin and gives a smooth field [1, Sec. 4.2.1]. `Comb` combines a smooth field at short distances with the exponential decay of 3GPP at large ones.

The ACF has a partner in the frequency domain: the ACF and the power spectral density of a stationary field are a Fourier pair (Wiener-Khinchin theorem) [1, Sec. 4.2.1]. The spectrum tells how much of the variation of the field comes from each spatial frequency, i.e. from waves of each length and direction. This is the key to the method below.

## How random fields are generated

There are two families of methods:

| Family | Idea | Examples | Strength | Cost |
|---|---|---|---|---|
| Matrix methods | Build the covariance matrix of all positions and factor it | Cholesky decomposition [9, Sec. 2.1] | Exact at the given positions | Memory grows with the square of the number of positions, time up to the cube; new positions need a new factorization |
| Sums of waves | Add up waves whose spectrum follows the target | FFT on a grid with circulant embedding [9, Sec. 2.2], randomization method [4, Sec. 2.2.2], sum of sinusoids [2] | Fast, little memory | ACF mostly approximate (circulant embedding is exact on its grid); FFT grids are periodic and bounded |

The sum-of-waves methods differ in three choices: where the frequencies come from (a regular grid, random draws, or a fit), whether the amplitudes are fixed or random, and whether the sum is evaluated by FFT on a grid or directly at each position. Quadriga-Lib uses fixed amplitudes, fitted frequencies and direct evaluation [2]. Each of its properties below follows from these choices.

## The sum-of-sinusoids method

A generator with $N$ sinusoids gives at position $\mathbf{x}$ the value

$$s(\mathbf{x}) = \sqrt{\frac{2}{N}} \sum_{n=1}^{N} \cos\!\left( 2\pi\, \mathbf{f}_n \cdot \mathbf{x} + \varphi_n \right)$$

[2], [11]. The frequency vectors $\mathbf{f}_n$ are samples of the spectrum: they set the ACF, which for random phases is $\frac{1}{N}\sum_n \cos(2\pi\, \mathbf{f}_n \cdot \mathbf{d})$. Quadriga-Lib fits them to the target ACF and ships the results as *presets*. The phases $\varphi_n$, drawn uniformly at random, select one realization. Each column of the phase array is one generator: columns with different phases give independent fields with the same ACF [2, Sec. II], so for one quantity they are its realizations. Since the value is a sum of many independent terms, it is close to Normal distributed (central limit theorem) [11].

The formula is short enough to evaluate by hand. The cell below does so with numpy for five random positions and compares the result with `sos.val`, the library function that evaluates a generator:

In [ ]:
pos = np.random.default_rng(1).uniform(0.0, 100.0, (5, 3))       # five positions {x, y, z} in m
N = freq.shape[0]
s_numpy = np.sqrt(2 / N) * np.cos(2 * np.pi * pos @ freq.T + phase[:, 0]).sum(axis=1)
s_lib = qd.sos.val(freq, phase, pos)[:, 0]
print(np.round(s_numpy, 4))
print(np.round(s_lib, 4))

With few sinusoids the values are not yet Normal distributed: a sum of $N$ cosines with amplitude $\sqrt{2/N}$ can never exceed $\sqrt{2N}$. The histograms show the values of 20 000 realizations at one position, for the first sinusoid alone, the first 4, and all 256 sinusoids of the preset. The density axis is logarithmic, so that the tails are visible: in the middle, 4 sinusoids already look Normal, but their values stop at $\sqrt{8} \approx 2.8$, where the Normal distribution goes on.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3), sharey=True)
v = np.linspace(-4.5, 4.5, 200)
for ax, n in zip(axs, (1, 4, 256)):
    ph = qd.sos.init_phases(n, 20000, 3)                           # 20 000 realizations
    vals = qd.sos.val(freq[:n], ph, np.zeros((1, 3))).ravel()        # value at the origin
    ax.hist(vals, bins=np.linspace(-4.5, 4.5, 61), density=True, alpha=0.6)
    ax.plot(v, np.exp(-v**2 / 2) / np.sqrt(2 * np.pi), "k", lw=1)
    ax.set_yscale("log"); ax.set_ylim(1e-4, 3)
    ax.set_title(f"N = {n}"); ax.set_xlabel("value")
axs[0].set_ylabel("density")
plt.show()

The method has clear strengths:

- Direct evaluation: the value at a position is computed from the formula alone. Unlike FFT methods, which compute the whole field on a grid of points and keep it in memory [2, Table I], any position can be evaluated at any time, in 1, 2 or 3 dimensions, without bounds.
- Little memory: a generator is $N$ frequency vectors and $N$ phases, independent of the size of the area.
- Repeatable: the same frequencies and phases always give the same field.
- Isotropic: the frequency vectors point in all directions.

And costs:

- The ACF is only approximated, and only over the distances the preset was fitted to (up to 5 $d_0$). Beyond that, the ACF of the generator does not fall to zero but keeps a small residual with rms $1/\sqrt{2N}$, 0.044 for 256 sinusoids.
- The computing time per position grows with $N$.
- The values are approximately, not exactly, Normal distributed.

## Rough and smooth fields

`sos.randn` is the shortest way to spatially correlated Normal values: it takes the decorrelation distances and the positions, builds the generators from a preset and draws their phases from the seed. Along a 100 m track, the exponential ACF gives a curve with more variation over short distances than the Gaussian ACF. The truly rough field of the exact exponential ACF would need infinitely many sinusoids; a preset reproduces its shape over the fitted distances, but not the kink at the origin [2, Sec. V]. Both curves start at the same value because they use the same seed and the same number of sinusoids (see "Repeatable simulations"). The values have zero mean and unit variance; a physical quantity is obtained by scaling, e.g. shadow fading in dB with a standard deviation of 6 dB as `6.0 * val`.

In [ ]:
track = np.zeros((1001, 3)); track[:, 0] = np.linspace(0.0, 100.0, 1001)     # 100 m along x, 10 cm steps

fig, ax = plt.subplots(figsize=(8, 3))
for name in ("Exp256", "Gauss256"):
    val = qd.sos.randn(np.array([10.0]), track, preset=name, seed=7)
    ax.plot(track[:, 0], val[:, 0], label=name)
ax.set_xlabel("position in m"); ax.set_ylabel("value"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, name in zip(axs, ("Exp256", "Gauss256")):
    f, a, d = qd.sos.preset(name, 10.0)
    m = qd.sos.map(f, qd.sos.init_phases(f.shape[0], 1, 3), x, x)[:, :, 0, 0]
    im = ax.imshow(m, origin="lower", extent=[0, 200, 0, 200], cmap=plt.get_cmap("viridis", 32), vmin=-3, vmax=3)
    ax.set_title(name); ax.set_xlabel("x in m")
axs[0].set_ylabel("y in m")
fig.colorbar(im, ax=axs, label="value", shrink=0.9)
plt.show()

## Choosing a preset

Presets combine an ACF shape (`Exp`, `Gauss`, `Comb`) with a number of sinusoids (128, 256, 512 or 1024), e.g. `"Comb256"`. More sinusoids approximate the ACF better and cost proportionally more time. The table gives the error of the ACF of each preset against its target, as an rms over distances and 200 directions, inside the fitted range (0 to 5 $d_0$) and beyond it:

| Preset | rms error, 0 to 5 $d_0$ | residual beyond 5 $d_0$ |
|---|---:|---:|
| `Exp128` / `Exp256` / `Exp512` / `Exp1024` | 0.041 / 0.028 / 0.019 / 0.013 | 0.063 / 0.044 / 0.032 / 0.024 |
| `Gauss128` / `Gauss256` / `Gauss512` / `Gauss1024` | 0.023 / 0.017 / 0.009 / 0.005 | 0.062 / 0.043 / 0.031 / 0.023 |
| `Comb128` / `Comb256` / `Comb512` / `Comb1024` | 0.028 / 0.016 / 0.012 / 0.008 | 0.063 / 0.045 / 0.032 / 0.024 |

The residual beyond the fitted range depends only on the number of sinusoids, $1/\sqrt{2N}$. `Comb256` is the default: smooth along a track, the exponential decay of 3GPP at large distances, and a good balance of accuracy and speed. Take `Exp` where the exponential shape is prescribed exactly, a larger preset where the shape of the ACF matters, a smaller one where speed does.

## Checking the correlation

The ACF of a generated field can be estimated from the field itself: take many pairs of positions, group them by their distance, and correlate the values within each group. `sos.acf_estimate` does this for one realization at random positions in a cube around the origin whose size follows the largest distance bin. The cube must span many decorrelation distances, and the estimate of one realization scatters, so the cell averages 20 realizations. For comparison it also evaluates the ACF of the generator itself from the formula above, $\frac{1}{N}\sum_n \cos(2\pi\, \mathbf{f}_n \cdot \mathbf{d})$, along the x axis:

In [ ]:
bins = np.arange(0.0, 250.5, 1.0)          # distance bins up to 250 m; the positions fill a cube of 0.8 × 250 m = 200 m edge
ph20 = qd.sos.init_phases(freq.shape[0], 20, 2)                    # 20 realizations
est = [qd.sos.acf_estimate(freq, ph20[:, r], bins, 1000, seed=r + 1) for r in range(20)]
acf_est = np.mean([e[0] for e in est], axis=0)
dist_est = est[0][1]
ok = np.isfinite(acf_est)                                            # bins without enough pairs are NaN
acf_gen_x = np.cos(2 * np.pi * np.outer(dist, freq[:, 0])).mean(axis=1)  # ACF of the generator along x

fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.plot(dist, acf, "k--", label="target (Comb)")
ax.plot(dist, acf_gen_x, "C1", label="generator (formula)")
ax.plot(dist_est[ok], acf_est[ok], "o", ms=3, label="estimated from the field")
ax.set_xlim(0, 50); ax.set_xlabel("distance d in m"); ax.set_ylabel("ACF"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
at10 = lambda d, a: np.interp(10.0, d, a)
print(f"ACF at 10 m: target {at10(dist, acf):.2f}, generator {at10(dist, acf_gen_x):.2f}, estimated {at10(dist_est[ok], acf_est[ok]):.2f}")

The estimate scatters around the ACF of the generator, and the generator follows the target closely; it differs most near $d_0$, by a few hundredths, where the `Comb` shape changes from `Gauss` to `Exp`. With a smaller cube, a few decorrelation distances across, the estimate falls below the target: the correlation within each bin is taken around the mean of that small region, which differs from the mean of the ensemble (see "Repeatable simulations" below). In two dimensions, the ACF of the generator itself can be compared with the target directly: `sos.acf_2d` gives the target on a grid of displacements, `sos.acf_approx` the ACF of the frequencies of a generator. Both are round: the field has the same correlation in every direction.

In [ ]:
acf_target, d2 = qd.sos.acf_2d(acf, dist)
acf_gen, _ = qd.sos.acf_approx(freq, dist)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, a, title in zip(axs, (acf_target, acf_gen[:, :, 0]), ("target", "generator, x-y plane")):
    im = ax.imshow(a, origin="lower", extent=[d2[0], d2[-1], d2[0], d2[-1]], cmap=plt.get_cmap("viridis", 32), vmin=-0.1, vmax=1)
    ax.set_xlim(-30, 30); ax.set_ylim(-30, 30); ax.set_title(title); ax.set_xlabel("dx in m")
axs[0].set_ylabel("dy in m")
fig.colorbar(im, ax=axs, label="ACF", shrink=0.9)
plt.show()

## Repeatable simulations

The seed selects the phases, and the phases select the field. A later call with the same seed, decorrelation distance and preset returns the same field, also at positions that were not part of the first call. A track can therefore be extended without changing the part already computed:

In [ ]:
first = qd.sos.randn(np.array([10.0]), track[:501], seed=11)       # first 50 m
whole = qd.sos.randn(np.array([10.0]), track, seed=11)             # all 100 m, in a later call
print("first 50 m unchanged:", np.array_equal(first, whole[:501]))

A seed selects a set of phases, not a field. Two calls with the same seed but different decorrelation distances use the same phases with scaled frequencies: the field for 20 m is the field for 10 m stretched by a factor of two. The two fields are equal at the origin and strongly correlated near it, which is wrong for quantities that must be independent, such as shadow fading and delay spread. One call with several decorrelation distances gives independent generators. The cell measures the correlation over 1000 seeds at four positions:

In [ ]:
p4 = np.array([[0.0, 0, 0], [5.0, 0, 0], [20.0, 0, 0], [100.0, 0, 0]])     # positions along x
two_calls, one_call = [], []
for s in range(1000):
    a = qd.sos.randn(np.array([10.0]), p4, seed=s)
    b = qd.sos.randn(np.array([20.0]), p4, seed=s)
    ab = qd.sos.randn(np.array([10.0, 20.0]), p4, seed=s)
    two_calls.append(np.c_[a, b]); one_call.append(ab)
two_calls, one_call = np.array(two_calls), np.array(one_call)        # [seed, position, generator]
for i, x0 in enumerate(p4[:, 0]):
    c2 = np.corrcoef(two_calls[:, i, 0], two_calls[:, i, 1])[0, 1]
    c1 = np.corrcoef(one_call[:, i, 0], one_call[:, i, 1])[0, 1]
    print(f"x = {x0:5.0f} m:  same seed, two calls {c2:+.2f}   one call {c1:+.2f}")

To keep a generator for later, store its frequencies and phases: `sos.preset` gives the frequencies, `sos.init_phases` the phases, and `sos.val` evaluates them at any positions, as in the first sections.

One realization is not the ensemble. The mean over a 100 m track of one realization is not zero; for an exponential ACF with $d_0 = 10$ m it scatters with a standard deviation of about 0.42 around zero. The variance of such local averages of a random field is a classic topic of random-field theory [12]. Many realizations show this:

In [ ]:
f_exp, _, _ = qd.sos.preset("Exp256", 10.0)
ph = qd.sos.init_phases(f_exp.shape[0], 2000, 21)                  # 2000 realizations
track_mean = qd.sos.val(f_exp, ph, track).mean(axis=0)            # mean of each realization over 100 m

fig, ax = plt.subplots(figsize=(6.4, 3))
ax.hist(track_mean, bins=40, alpha=0.7)
ax.set_xlabel("mean over the 100 m track"); ax.set_ylabel("number of realizations")
plt.show()
print(f"standard deviation of the track mean: {track_mean.std():.2f}")

A simulation that averages over the users of one area therefore needs many realizations (drops), or an area that spans many decorrelation distances.

## Uniform and integer values

`sos.rand` gives spatially correlated values that are uniform between 0 and 1, `sos.randi` integers from 1 to a maximum. A threshold on uniform values gives a map of two states: with a threshold of 0.3, each position is "blocked" with a probability of 30 %. This is how spatially consistent line-of-sight states are drawn in the 3GPP channel model [3, clause 7.6.3.3], and how two-phase materials are modeled [7]. Integer values give regions of a few classes:

In [ ]:
gx, gy = np.meshgrid(x, x)
grid = np.c_[gx.ravel(), gy.ravel(), np.zeros(gx.size)]

blocked = qd.sos.rand(np.array([10.0]), grid, seed=3)[:, 0] < 0.3          # 30 % blocked
classes = qd.sos.randi(np.array([10.0]), grid, 4, seed=9)[:, 0]          # classes 1 to 4

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
axs[0].imshow(blocked.reshape(gx.shape), origin="lower", extent=[0, 200, 0, 200], cmap="gray_r")
axs[0].set_title(f"blocked: {blocked.mean():.0%} of the area")
im = axs[1].imshow(classes.reshape(gx.shape), origin="lower", extent=[0, 200, 0, 200], cmap=plt.get_cmap("viridis", 4), vmin=0.5, vmax=4.5)
axs[1].set_title("classes 1 to 4")
fig.colorbar(im, ax=axs[1], ticks=[1, 2, 3, 4], label="class", shrink=0.9)
for ax in axs: ax.set_xlabel("x in m")
axs[0].set_ylabel("y in m")
plt.show()

The share of the area in this one map is 33 %, not 30 %, for the same reason as the track mean above: one map spans only 20 decorrelation distances, and the 30 % hold for the ensemble. Over many maps of this size, the share scatters by about 5 percentage points (one standard deviation). The classes of `sos.randi` are value bands of one field, so neighboring regions have neighboring classes.

## Comparison with a matrix method

The Cholesky method from the table above is exact: it builds the covariance matrix of all positions and factors it. With numpy this takes a few lines for 400 random positions:

In [ ]:
rng = np.random.default_rng(12)
p400 = np.c_[rng.uniform(0.0, 60.0, (400, 2)), np.zeros(400)]
D = np.linalg.norm(p400[:, None, :] - p400[None, :, :], axis=2)    # distances between all positions
L = np.linalg.cholesky(np.exp(-D / 10.0))                          # exponential ACF, d0 = 10 m
v_chol = L @ rng.standard_normal(400)
v_sos = qd.sos.randn(np.array([10.0]), p400, preset="Exp256", seed=12)[:, 0]

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, v, title in zip(axs, (v_chol, v_sos), ("Cholesky", "sum of sinusoids (Exp256)")):
    sc = ax.scatter(p400[:, 0], p400[:, 1], c=v, cmap="viridis", s=30, vmin=-2.5, vmax=2.5)
    ax.set_title(title); ax.set_xlabel("x in m")
axs[0].set_ylabel("y in m")
fig.colorbar(sc, ax=axs, label="value", shrink=0.9)
plt.show()
print(f"covariance matrix: {D.nbytes / 1e6:.1f} MB for 400 positions, {8 * 100_000**2 / 1e9:.0f} GB for 100 000")

Both fields follow the same target ACF: the Cholesky field exactly at these positions, the sum-of-sinusoids field approximately (it smooths the kink of the exponential ACF at the origin, and its values are only approximately Normal). The main difference is the cost: the matrix grows with the square of the number of positions, its factorization with the cube [9, Sec. 2.1], and every new position needs a new factorization. The sum-of-sinusoids generator needs only its frequencies and phases and evaluates any new position directly, which makes it suitable for long tracks, large areas and many users.

## Summary

- A random field is described by its ACF; the decorrelation distance $d_0$ is where the ACF falls to $1/e$.
- The sum-of-sinusoids method adds sinusoids with fitted frequencies and random phases. It needs no grid, little memory, and evaluates any position at any time.
- Presets combine an ACF shape with a number of sinusoids; `Comb256` is the default.
- A seed fixes the phases and makes a simulation repeatable; independent quantities need independent generators.
- One realization is not the ensemble: averages need many realizations.

The next tutorial shows how to fit the frequencies to an ACF of your own, and where the method reaches its limits.

## References

[1] C. E. Rasmussen, C. K. I. Williams, *Gaussian Processes for Machine Learning*, MIT Press, 2006, chapters 2 and 4. [Free online](https://gaussianprocess.org/gpml/chapters/).

[2] S. Jaeckel, L. Raschkowski, F. Burkhardt, L. Thiele, "Efficient sum-of-sinusoids-based spatial consistency for the 3GPP new-radio channel model", IEEE Globecom Workshops, 2018. [PDF](publications/2018_sum_of_sinusoids.pdf), [doi:10.1109/GLOCOMW.2018.8644265](https://doi.org/10.1109/GLOCOMW.2018.8644265).

[3] 3GPP TR 38.901 V19.5.0 (2026-09), "Study on channel model for frequencies from 0.5 to 100 GHz", clauses 7.4.4 (exponential ACF of the shadow fading), 7.5 (Table 7.5-6, correlation distances) and 7.6.3 (spatial consistency). [Online](https://www.3gpp.org/ftp/Specs/archive/38_series/38.901/).

[4] S. Müller, L. Schüler, A. Zech, F. Heße, "GSTools v1.3: a toolbox for geostatistical modelling in Python", Geosci. Model Dev. 15, 3161-3182, 2022. [doi:10.5194/gmd-15-3161-2022](https://doi.org/10.5194/gmd-15-3161-2022).

[5] J. Tessendorf, "Simulating ocean water", course notes, 1999-2004. [PDF](https://jtessen.people.clemson.edu/reports/papers_files/coursenotes2004.pdf).

[6] M. Charnotskii, "Four methods for generation of turbulent phase screens: comparison", 2019. [arXiv:1911.09185](https://arxiv.org/abs/1911.09185). Published as "Comparison of four techniques for turbulent phase screens simulation", J. Opt. Soc. Am. A 37(5), 738, 2020, [doi:10.1364/JOSAA.385754](https://doi.org/10.1364/JOSAA.385754).

[7] A. P. Roberts, M. Teubner, "Transport properties of heterogeneous materials derived from Gaussian random fields: bounds and simulation", Phys. Rev. E 51, 4141-4154, 1995. [doi:10.1103/PhysRevE.51.4141](https://doi.org/10.1103/PhysRevE.51.4141).

[8] M. Gudmundson, "Correlation model for shadow fading in mobile radio systems", Electronics Letters 27(23), 2145-2146, 1991. [doi:10.1049/el:19911328](https://doi.org/10.1049/el:19911328).

[9] D. P. Kroese, Z. I. Botev, "Spatial process generation", 2013. [arXiv:1308.0399](https://arxiv.org/abs/1308.0399). Published as "Spatial process simulation", Lecture Notes in Mathematics 2120, Springer, 2015, pp. 369-404, [doi:10.1007/978-3-319-10064-7_12](https://doi.org/10.1007/978-3-319-10064-7_12).

[10] M. Shinozuka, G. Deodatis, "Simulation of stochastic processes by spectral representation", Appl. Mech. Rev. 44(4), 191-204, 1991. [doi:10.1115/1.3119501](https://doi.org/10.1115/1.3119501).

[11] S. O. Rice, "Mathematical analysis of random noise" (Parts I and II), Bell Syst. Tech. J. 23(3), 282-332, 1944. [doi:10.1002/j.1538-7305.1944.tb00874.x](https://doi.org/10.1002/j.1538-7305.1944.tb00874.x).

[12] E. Vanmarcke, *Random Fields: Analysis and Synthesis*, revised edition, World Scientific, 2010. [doi:10.1142/5807](https://doi.org/10.1142/5807).

[13] GSTools documentation, covariance models (length scale and integral scale). [Online](https://geostat-framework.readthedocs.io/projects/gstools/en/stable/api/gstools.covmodel.Gaussian.html).

[14] Z. Wang, E. K. Tameh, A. R. Nix, "A sum-of-sinusoids based simulation model for the joint shadowing process in urban peer-to-peer radio channels", IEEE VTC 2005-Fall, vol. 3, pp. 1732-1736. [doi:10.1109/VETECF.2005.1558401](https://doi.org/10.1109/VETECF.2005.1558401).

For further reading: [1] chapter 4 for the theory of ACFs and spectra; M. Pätzold, *Mobile Radio Channels*, 2nd ed., Wiley, 2011, for sum-of-sinusoids models in radio channels; [10] for the spectral representation method in engineering.